# 3 — RAG corpus ingestion (Vertex AI RAG Engine)

**AWS-side equivalent:** Titan Embeddings V2 + pgvector storage in Aurora
**GCP service:** Vertex AI RAG Engine — corpus + ingestion as a managed
service. Embeddings are computed internally by the corpus (we do **not**
call `embed_content` ourselves the way the AWS notebook does).

AWS-side notebook: `notebooks/aws/03-embeddings.ipynb`

> **API note (verified May 2026):** `vertexai.rag` is GA and is the path forward.
> The companion `vertexai.generative_models` namespace deprecates **2026-06-24**;
> for now we use it in `04b-rag-engine.ipynb` for the `Tool.from_retrieval`
> wiring, and will migrate to the unified `google-genai` SDK once it ships
> the equivalent surface.
>
> **Embedding model constraint:** RAG Engine only supports `text-embedding-005`
> or `text-embedding-004`. The corpus-managed embedding stack does **not**
> accept `gemini-embedding-001` (despite our raw `embed_content` notebooks
> using that model). We pin to `text-embedding-005` here.


## Step 0: Imports + config

In [18]:
import json
from pathlib import Path

import vertexai
from vertexai import rag
from google.cloud import bigquery, storage
from google.api_core.exceptions import Conflict, NotFound

from cleanup_utils import ResourceTracker

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})

PROJECT_ID    = gcp.get('project_id')
REGION        = gcp.get('region', 'us-central1')
BQ_DATASET    = gcp.get('bq_dataset', 'orgiq')
STAGING_BUCKET = gcp.get('staging_bucket', f'gs://{PROJECT_ID}-orgiq-staging')
CORPUS_NAME   = gcp.get('rag_corpus_display_name', 'orgiq-people-corpus')
CHUNK_SIZE    = gcp.get('rag_chunk_size', 512)
CHUNK_OVERLAP = gcp.get('rag_chunk_overlap', 100)

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

vertexai.init(project=PROJECT_ID, location=REGION)
tracker = ResourceTracker()

print(f'project        : {PROJECT_ID}')
print(f'region         : {REGION}')
print(f'BQ dataset     : {BQ_DATASET}')
print(f'staging bucket : {STAGING_BUCKET}')
print(f'corpus display : {CORPUS_NAME}')
print(f'chunk size     : {CHUNK_SIZE} (overlap {CHUNK_OVERLAP})')

project        : hoehncloudconsole
region         : us-central1
BQ dataset     : orgiq
staging bucket : gs://hoehncloudconsole-orgiq-staging
corpus display : orgiq-people-corpus
chunk size     : 512 (overlap 100)


## Step 1: Read people from BigQuery

Mirrors the AWS-side query against `people` in Aurora. If the BigQuery
table is missing (e.g. notebook 02 hasn't run), fall back to the local
seed file so this notebook still works in isolation.

In [19]:
def load_people() -> list[dict]:
    """Try BigQuery first, fall back to the local seed file."""
    try:
        bq = bigquery.Client(project=PROJECT_ID)
        query = f"""
            SELECT id, name, title, department, manager_id
            FROM `{PROJECT_ID}.{BQ_DATASET}.people`
        """
        rows = [dict(r) for r in bq.query(query).result()]
        if rows:
            print(f'✓ Loaded {len(rows)} people from BigQuery')
            return rows
        print('  BigQuery returned 0 rows; falling back to seed file')
    except Exception as exc:  # noqa: BLE001
        print(f'  BigQuery read failed ({type(exc).__name__}); using seed file')

    seed = json.loads(Path('data/org/people.json').read_text())
    print(f'✓ Loaded {len(seed)} people from data/org/people.json')
    return seed

people = load_people()
people[0]

✓ Loaded 20 people from BigQuery


{'id': 'p-005',
 'name': 'David Kim',
 'title': 'Engineering Manager',
 'department': 'Platform Engineering',
 'manager_id': None}

## Step 2: Compose person-as-document markdown

One markdown file per person — RAG Engine handles chunking + embedding
internally during `import_files`. The corpus will end up with `N` source
files and (post-chunking) some multiple of that as retrievable nodes.

We also load `teams.json` so we can put each person on their team and
mention the team's charter — that gives the retriever some semantic
ballast beyond just job titles.

In [20]:
# Cross-reference teams so we can write "{name} is on the {team} team" lines.
teams_raw = json.loads(Path('data/org/teams.json').read_text())
team_by_member = {}
for t in teams_raw:
    for member_id in t.get('member_ids', []):
        team_by_member[member_id] = t

def person_to_markdown(p: dict) -> str:
    team = team_by_member.get(p['id'])
    lines = [
        f"# {p['name']}",
        '',
        f"**Title:** {p.get('title', 'unknown')}",
        f"**Department:** {p.get('department', 'unknown')}",
    ]
    if p.get('manager_id'):
        lines.append(f"**Manager:** {p['manager_id']}")
    if team:
        lines += [
            '',
            f"## Team — {team['name']}",
            '',
            team.get('charter', ''),
        ]
    lines += [
        '',
        '## Role summary',
        '',
        f"{p['name']} is a {p.get('title', 'team member')} in {p.get('department', 'the org')}. "
        + (f"Works on the {team['name']} team. " if team else '')
        + 'Reach out for questions about their domain.',
    ]
    return '\n'.join(lines).strip() + '\n'

# Write the JSONL ledger + per-person markdown to /tmp.
jsonl_path = Path('/tmp/orgiq-people.jsonl')
md_dir = Path('/tmp/orgiq-people-md')
md_dir.mkdir(exist_ok=True)

with jsonl_path.open('w') as f:
    for p in people:
        body = person_to_markdown(p)
        (md_dir / f"{p['id']}.md").write_text(body)
        f.write(json.dumps({'id': p['id'], 'name': p['name'], 'body': body}) + '\n')

print(f'✓ Wrote {len(people)} markdown files to {md_dir}/')
print(f'✓ Wrote JSONL ledger to {jsonl_path}')
print('\n--- sample ---')
print((md_dir / f"{people[0]['id']}.md").read_text())

✓ Wrote 20 markdown files to /tmp/orgiq-people-md/
✓ Wrote JSONL ledger to /tmp/orgiq-people.jsonl

--- sample ---
# David Kim

**Title:** Engineering Manager
**Department:** Platform Engineering

## Team — Platform Engineering

Build and maintain core platform infrastructure including APIs, authentication, CI/CD pipelines, and developer tooling.

## Role summary

David Kim is a Engineering Manager in Platform Engineering. Works on the Platform Engineering team. Reach out for questions about their domain.



## Step 3: Upload to the GCS staging bucket

The corpus reads files from GCS. We use a stable prefix `orgiq-corpus/people/`
inside the staging bucket so 19-cleanup can purge it cheaply (deleting the
corpus drops the embeddings but leaves the source files in place).

In [21]:
# STAGING_BUCKET may be a bare name or a gs://… URI — normalize.
bucket_name = STAGING_BUCKET[5:].split('/', 1)[0] if STAGING_BUCKET.startswith('gs://') else STAGING_BUCKET

gcs = storage.Client(project=PROJECT_ID)
try:
    bucket = gcs.create_bucket(bucket_name, location='US')
    print(f'✓ Created bucket gs://{bucket_name}')
except Conflict:
    bucket = gcs.bucket(bucket_name)
    print(f'  Bucket gs://{bucket_name} already exists')

tracker.track('gcs_bucket', bucket_name)
tracker.flush()

prefix = 'orgiq-corpus/people'
for md_file in sorted(md_dir.glob('*.md')):
    blob = bucket.blob(f'{prefix}/{md_file.name}')
    blob.upload_from_string(md_file.read_text(), content_type='text/markdown')

print(f'✓ Uploaded {len(list(md_dir.glob("*.md")))} files to gs://{bucket_name}/{prefix}/')
CORPUS_GCS_URI = f'gs://{bucket_name}/{prefix}/'
print(f'  corpus source URI: {CORPUS_GCS_URI}')

  Bucket gs://hoehncloudconsole-orgiq-staging already exists
✓ Uploaded 20 files to gs://hoehncloudconsole-orgiq-staging/orgiq-corpus/people/
  corpus source URI: gs://hoehncloudconsole-orgiq-staging/orgiq-corpus/people/


## Step 4: Create (or reuse) the RAG corpus

`rag.create_corpus` does **not** dedupe on `display_name` — calling it
twice creates two corpora. We list first, reuse on hit, create on miss.

**Why REST instead of SDK:** `google-cloud-aiplatform==1.153.1` has a broken
middle state for RAG corpus creation on new projects:

- Passing any `RagVectorDbConfig` (even the `RagManagedDb` serverless variant)
  triggers Spanner mode, which is allowlist-only for new projects in
  `us-central1` / `us-east1` / `us-east4`.
- Calling without `backend_config` also routes to Spanner by default.
- The `rag_managed_db` proto field needed to force Serverless mode exists in the
  REST API but is **missing from the bundled GAPIC proto** in this SDK version —
  so `RagManagedDb()` raises `AttributeError: Unknown field for RagManagedDb`.

Workaround: POST to the Vertex AI REST endpoint directly with
`rag_vector_db_config: {rag_managed_db: {}}` to force Serverless mode, then
poll the returned LRO until done, and look up the new corpus via `rag.list_corpora`.


In [ ]:
import time
import requests
import google.auth
import google.auth.transport.requests


def find_or_create_corpus(display_name: str):
    # Reuse if a corpus with this display_name already exists.
    for c in rag.list_corpora():
        if c.display_name == display_name:
            print(f'  Reusing existing corpus: {c.name}')
            return c

    # SDK path is broken for Serverless mode on new projects (see Step 4 note).
    # Use REST API directly: POST with rag_managed_db: {} forces Serverless mode.
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_header = {'Authorization': f'Bearer {creds.token}'}

    url = (
        f'https://{REGION}-aiplatform.googleapis.com/v1'
        f'/projects/{PROJECT_ID}/locations/{REGION}/ragCorpora'
    )
    body = {
        'display_name': display_name,
        'rag_vector_db_config': {'rag_managed_db': {}},
    }
    resp = requests.post(url, json=body, headers=auth_header)
    resp.raise_for_status()

    # Poll the returned LRO until done (typically < 60 s).
    op_name = resp.json()['name']
    op_url = f'https://{REGION}-aiplatform.googleapis.com/v1/{op_name}'
    for _ in range(60):
        op = requests.get(op_url, headers=auth_header).json()
        if op.get('done'):
            break
        time.sleep(5)
    else:
        raise TimeoutError('RAG corpus creation timed out after 5 min')

    corpus_name = op['response']['name']
    print(f'✓ Created corpus: {corpus_name}')

    # Return the SDK object so the rest of the notebook works unchanged.
    for c in rag.list_corpora():
        if c.name == corpus_name:
            return c
    raise RuntimeError(f'Corpus created but not found in list: {corpus_name}')


corpus = find_or_create_corpus(CORPUS_NAME)

tracker.track('rag_corpus', corpus.name, project=PROJECT_ID, location=REGION)
tracker.flush()

# Persist the full resource name so 04b can find it.
gcp['rag_corpus_name'] = corpus.name
config_path.write_text(json.dumps(config, indent=2))
print(f'  config.gcp.rag_corpus_name = {corpus.name}')


## Step 5: Import the staged files into the corpus

`rag.import_files` returns synchronously once the LRO finishes (under the
hood it polls for us). For the workshop's 20-person corpus this is
typically under 2 minutes. The `max_embedding_requests_per_min` knob
keeps us safely under the embedding-model quota.

In [ ]:
import_response = rag.import_files(
    corpus.name,
    paths=[CORPUS_GCS_URI],
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        ),
    ),
    max_embedding_requests_per_min=1000,
)

# `import_response` exposes per-file counts.
print('✓ Import complete')
for attr in ('imported_rag_files_count', 'failed_rag_files_count', 'skipped_rag_files_count'):
    val = getattr(import_response, attr, None)
    if val is not None:
        print(f'  {attr}: {val}')

## Step 6: Verify — list files in the corpus

In [ ]:
files = list(rag.list_files(corpus.name))
print(f'Corpus contains {len(files)} files:')
for f in files[:5]:
    # `display_name` typically matches the source filename.
    print(f'  - {f.display_name}  ({f.name.split("/")[-1]})')
if len(files) > 5:
    print(f'  … and {len(files) - 5} more')

## Summary

| Resource | Details |
|---|---|
| **Service** | Vertex AI RAG Engine (managed retrieval) |
| **Corpus** | `rag.create_corpus(display_name='orgiq-people-corpus')` |
| **Backend** | Default `RagManagedDb` — no Vector Search provisioning |
| **Embeddings** | `text-embedding-005` (managed by the corpus) |
| **Chunking** | size=512 tokens, overlap=100 |
| **Source** | `gs://<staging>/orgiq-corpus/people/*.md` (one file per person) |
| **Idle cost** | Storage + per-query at retrieval time; no provisioning fee |
| **Tracked** | `rag_corpus` (Python deleter), `gcs_bucket` (gcloud CLI deleter) |

**Next:** `04-vertex-search.ipynb` covers turnkey Discovery Engine retrieval, and
`04b-rag-engine.ipynb` uses *this* corpus for managed retrieval + Gemini grounded answers.
